# Model Inspection & Portability Guide (.joblib to other formats)

This notebook serves two main purposes:
1. **Inspect Model Contents**: Load and print parameters, shapes, classes, and properties of our trained `.joblib` objects.
2. **Export / Convert Models**: Provide code samples to transition from joblib serialization to other runtime environments (such as **ONNX** or **JSON** export).

## 1. Load and Inspect the Joblib Files

First, we initialize directory paths and import `joblib`. We will load each trained object from our `data/models/` directory.

In [ ]:
import os
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

# Set up directories relative to the notebook path
NOTEBOOK_DIR = Path(os.getcwd())
ROOT_DIR = NOTEBOOK_DIR.parent
MODELS_DIR = ROOT_DIR / "data" / "models"

print(f"Loading models from: {MODELS_DIR}")

In [ ]:
# Load all artifacts
scaler = joblib.load(MODELS_DIR / "scaler.joblib")
categorical_encoders = joblib.load(MODELS_DIR / "categorical_encoders.joblib")
category_label_encoder = joblib.load(MODELS_DIR / "category_label_encoder.joblib")
feature_columns = joblib.load(MODELS_DIR / "feature_columns.joblib")

rf_binary = joblib.load(MODELS_DIR / "rf_binary.joblib")
rf_category = joblib.load(MODELS_DIR / "rf_category.joblib")

email_vectorizer = joblib.load(MODELS_DIR / "email_tfidf_vectorizer.joblib")
email_classifier = joblib.load(MODELS_DIR / "email_classifier.joblib")

autoencoder_meta = joblib.load(MODELS_DIR / "autoencoder_meta.joblib")

print("All joblib artifacts successfully loaded into memory!")

### 1.1 Inspect Scaler & Encoders

In [ ]:
print("=== scaler.joblib ===")
print(f"Type: {type(scaler)}")
print(f"Mean vector shape: {scaler.mean_.shape}")
print(f"Scale factor vector shape: {scaler.scale_.shape}")

print("\n=== categorical_encoders.joblib ===")
print(f"Keys: {list(categorical_encoders.keys())}")
for col, enc in categorical_encoders.items():
    print(f"  Encoder '{col}' class count: {len(enc.classes_)}")
    print(f"  Classes: {list(enc.classes_[:5])}... (truncated)")

print("\n=== category_label_encoder.joblib ===")
print(f"Target classification classes: {list(category_label_encoder.classes_)}")

### 1.2 Inspect ML Classifiers

In [ ]:
print("=== rf_binary.joblib (Random Forest) ===")
print(f"Number of estimators (Trees): {rf_binary.n_estimators}")
print(f"Number of features trained on: {rf_binary.n_features_in_}")
print(f"Classes: {rf_binary.classes_} (0=Normal, 1=Attack)")

print("\n=== rf_category.joblib (Random Forest Multi-class) ===")
print(f"Number of estimators (Trees): {rf_category.n_estimators}")
print(f"Number of features trained on: {rf_category.n_features_in_}")
print(f"Classes: {rf_category.classes_}")

In [ ]:
print("=== email_tfidf_vectorizer.joblib ===")
print(f"Vocabulary size: {len(email_vectorizer.vocabulary_)}")
print(f"N-gram range: {email_vectorizer.ngram_range}")
print(f"Stop words setting: {email_vectorizer.stop_words}")

print("\n=== email_classifier.joblib (Logistic Regression) ===")
print(f"Coefficients shape: {email_classifier.coef_.shape}")
print(f"Intercept: {email_classifier.intercept_[0]:.4f}")

## 2. Running Inference In-Notebook

In [ ]:
# Example 1: Email Phishing Inference
test_subject = "URGENT: Reset your banking credentials immediately"
test_body = "We have detected suspicious log in attempts. Go to http://scam-bank.com/login and verify your identity."
test_text = f"{test_subject} {test_body}"

# Vectorize
vectorized_text = email_vectorizer.transform([test_text])

# Predict
is_phishing = email_classifier.predict(vectorized_text)[0]
probability = email_classifier.predict_proba(vectorized_text)[0][1]

print(f"Input Text: \"{test_subject}\"")
print(f"Is Phishing? {bool(is_phishing)}")
print(f"Phishing Probability: {probability:.4f}")

In [ ]:
# Example 2: Network Traffic Data Preparation & Prediction
raw_connection = {
    "duration": 0,
    "protocol_type": "tcp",
    "service": "http",
    "flag": "SF",
    "src_bytes": 350,
    "dst_bytes": 1200,
    "land": 0,
    "wrong_fragment": 0,
    "urgent": 0,
    "hot": 0,
    "num_failed_logins": 0,
    "logged_in": 1,
    "num_compromised": 0,
    "root_shell": 0,
    "su_attempted": 0,
    "num_root": 0,
    "num_file_creations": 0,
    "num_shells": 0,
    "num_access_files": 0,
    "num_outbound_cmds": 0,
    "is_host_login": 0,
    "is_guest_login": 0,
    "count": 12,
    "srv_count": 12,
    "serror_rate": 0.0,
    "srv_serror_rate": 0.0,
    "rerror_rate": 0.0,
    "srv_rerror_rate": 0.0,
    "same_srv_rate": 1.0,
    "diff_srv_rate": 0.0,
    "srv_diff_host_rate": 0.0,
    "dst_host_count": 100,
    "dst_host_srv_count": 255,
    "dst_host_same_srv_rate": 1.0,
    "dst_host_diff_srv_rate": 0.0,
    "dst_host_same_src_port_rate": 0.01,
    "dst_host_srv_diff_host_rate": 0.02,
    "dst_host_serror_rate": 0.0,
    "dst_host_srv_serror_rate": 0.0,
    "dst_host_rerror_rate": 0.0,
    "dst_host_srv_rerror_rate": 0.0
}

# 1. Handle Categoricals
encoded = {}
for col in ["protocol_type", "service", "flag"]:
    raw_val = raw_connection.get(col, "UNK")
    enc = categorical_encoders[col]
    encoded[col + "_enc"] = enc.transform([raw_val])[0]

# 2. Vectorize according to feature columns
vector = []
for col in feature_columns:
    if col in encoded:
        vector.append(encoded[col])
    else:
        vector.append(float(raw_connection.get(col, 0)))

X_raw = np.array(vector, dtype=float).reshape(1, -1)
X_scaled = scaler.transform(X_raw)

# 3. Predict with RandomForest models
is_attack = rf_binary.predict(X_scaled)[0]
attack_prob = rf_binary.predict_proba(X_scaled)[0][1]

cat_idx = rf_category.predict(X_scaled)[0]
category = category_label_encoder.classes_[cat_idx]

print(f"Connection Classification:")
print(f"  Is Attack? {bool(is_attack)}")
print(f"  Attack Probability: {attack_prob:.4f}")
print(f"  Classified Category: {category}")

## 3. Export / Convert Models to Interoperable Formats

`.joblib` is highly coupled to Python, matching class signatures, and specific versions of scikit-learn. To execute these models inside environments like C++, C#, Java, Go, or client-side Javascript, we can export them to portable formats.

### 3.1 Exporting Models to ONNX Format (Standard Model Representation)

ONNX (Open Neural Network Exchange) is the industry standard for framework-agnostic model serialization. You can run ONNX models with `onnxruntime` in almost any programming language.

To do this, we use the `skl2onnx` library. Here is how you can convert our Logistic Regression email classifier to ONNX format:

In [ ]:
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    import onnx
    
    # Define the input type (TF-IDF produces float vectors of size 20000)
    initial_type = [('float_input', FloatTensorType([None, 20000]))]
    
    # Convert to ONNX
    onnx_model = convert_sklearn(email_classifier, initial_types=initial_type)
    
    # Save the model
    onnx_path = MODELS_DIR / "email_classifier.onnx"
    with open(onnx_path, "wb") as f:
        f.write(onnx_model.SerializeToString())
    print(f"Saved ONNX model successfully to {onnx_path}!")
except ImportError:
    print("To run this cell, install conversion tools:")
    print("  pip install skl2onnx onnxruntime onnx")

### 3.2 Exporting Models to JSON / Dict (Zero-Dependency Custom Inference)

For simple linear/logistic regression models or tree structures, we can serialize the model weights directly to JSON. This avoids external runtimes entirely.

Below, we extract the parameters of our Logistic Regression model and save it as a JSON file, which can be easily parsed by any language (e.g. NodeJS frontend, Go API, etc.).

In [ ]:
import json

# Extract parameters from Logistic Regression
model_params = {
    "coefficients": email_classifier.coef_[0].tolist(),
    "intercept": float(email_classifier.intercept_[0]),
    "classes": email_classifier.classes_.tolist()
}

json_path = MODELS_DIR / "email_classifier_parameters.json"
with open(json_path, "w") as f:
    json.dump(model_params, f, indent=2)

print(f"Saved model coefficients to JSON successfully: {json_path}")

Here is how a custom prediction function in Javascript or Python could run inference directly using the exported JSON parameters:

$$\text{probability} = \frac{1}{1 + e^{-(\sum w_i x_i + b)}}$$

In [ ]:
def predict_from_json(vectorized_input_features, json_model_path):
    # Load weights from JSON
    with open(json_model_path, "r") as f:
        weights = json.load(f)
        
    coef = np.array(weights["coefficients"])
    intercept = weights["intercept"]
    
    # Compute decision function
    # For sparse TF-IDF outputs, we compute dot product
    raw_score = (vectorized_input_features * coef)[0] + intercept
    
    # Apply Sigmoid function
    probability = 1.0 / (1.0 + np.exp(-raw_score))
    
    return probability

# Test the JSON-based predictor against the scikit-learn output
json_prob = predict_from_json(vectorized_text, json_path)
print(f"Scikit-learn predict_proba: {probability:.6f}")
print(f"Custom JSON-based predict:    {json_prob:.6f}")